In [8]:
# ==========================================
# 📊 PRODUCT SEGMENTATION (Fixed & Improved)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print("📊 Loading and preprocessing product data...")

# --- Load product dataset ---
df = pd.read_csv(r"C:\Users\Araya\Desktop\segementation Data enchnaced\product_features_enhanced (2).csv")
print(f"✅ Loaded dataset: {df.shape[0]:,} products, {df.shape[1]} features")

# --- Check available columns ---
print(f"\n📋 Available columns: {list(df.columns)}")

# --- Select core features (using available columns) ---
# Check which features exist in the dataset
potential_features = [
    "TotalRevenue", "TotalProfit", "ProfitMargin", 
    "SalesQty_sum", "ExtPrice_mean", "AvgOrderValue",
    "OrderNumber_nunique", "QuoteConversionRate"
]

# Only use features that actually exist in the dataset
selected_features = [f for f in potential_features if f in df.columns]
print(f"\n🎯 Selected features for clustering: {selected_features}")

if len(selected_features) < 3:
    print("❌ Error: Not enough features available for clustering!")
    print("Available numeric columns:")
    print(list(df.select_dtypes(include=[np.number]).columns))
else:
    # --- Extract feature matrix ---
    X = df[selected_features].copy()
    
    # --- Data Quality Check ---
    print(f"\n🔍 Data Quality Check:")
    print(f"   Original shape: {X.shape}")
    print(f"   Missing values: {X.isnull().sum().sum()}")
    print(f"   Infinite values: {np.isinf(X.values).sum()}")
    
    # --- Handle missing and infinite values ---
    # Replace infinite values with NaN first
    X = X.replace([np.inf, -np.inf], np.nan)
    
    # Check missing values per column
    missing_percent = (X.isnull().sum() / len(X)) * 100
    print(f"\n📊 Missing values by feature:")
    for feature, percent in missing_percent.items():
        print(f"   {feature}: {percent:.1f}%")
    
    # Remove features with too many missing values (>50%)
    good_features = missing_percent[missing_percent <= 50].index.tolist()
    X = X[good_features]
    selected_features = good_features
    
    print(f"\n✅ Final features after cleaning: {selected_features}")
    
    # --- Impute remaining missing values ---
    imputer = SimpleImputer(strategy='median')  # Use median to be robust to outliers
    X_imputed = pd.DataFrame(
        imputer.fit_transform(X), 
        columns=X.columns, 
        index=X.index
    )
    
    print(f"✅ Missing values after imputation: {X_imputed.isnull().sum().sum()}")
    
    # --- Remove extreme outliers (beyond 3 standard deviations) ---
    z_scores = np.abs((X_imputed - X_imputed.mean()) / X_imputed.std())
    outlier_mask = (z_scores < 3).all(axis=1)
    X_clean = X_imputed[outlier_mask]
    
    outliers_removed = len(X_imputed) - len(X_clean)
    print(f"🧹 Removed {outliers_removed:,} extreme outliers ({outliers_removed/len(X_imputed)*100:.1f}%)")
    print(f"📊 Clean dataset: {X_clean.shape}")
    
    # --- Standardize data ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clean)
    
    print(f"✅ Data standardized successfully")
    print(f"📈 Scaled data shape: {X_scaled.shape}")
    print(f"🔍 Scaled data stats: mean≈{X_scaled.mean():.3f}, std≈{X_scaled.std():.3f}")

📊 Loading and preprocessing product data...
✅ Loaded dataset: 40,359 products, 62 features

📋 Available columns: ['ProductID', 'ExtPrice_sum', 'ExtPrice_mean', 'ExtPrice_std', 'ExtPrice_count', 'ExtPrice_median', 'ExtPrice_min', 'ExtPrice_max', 'ExtCost_sum', 'ExtCost_mean', 'ExtCost_std', 'ExtCost_median', 'SalesQty_sum', 'SalesQty_mean', 'SalesQty_std', 'SalesQty_median', 'SalesQty_min', 'SalesQty_max', 'UnitPrice_mean_x', 'UnitPrice_std_x', 'UnitPrice_min_x', 'UnitPrice_max_x', 'UnitPrice_median', 'UnitCost_mean', 'UnitCost_std', 'UnitCost_median', 'OrderNumber_nunique', 'CustomerID_nunique', 'SalesDate_min', 'SalesDate_max', 'SalesDate_count', 'TotalRevenue', 'TotalCost', 'TotalProfit', 'ProfitMargin', 'AvgOrderValue', 'AvgOrderQuantity', 'PriceVolatility', 'QuantityVolatility', 'ProductLifetimeDays', 'AvgDaysBetweenOrders', 'SalesFrequency', 'CustomerBase', 'CustomerPenetration', 'RevenuePerCustomer', 'UnitsPerCustomer', 'OrdersPerCustomer', 'QuoteID_nunique', 'QuoteVersion_mean',

In [ ]:
# ==========================================
# 🔍 FIND THE OPTIMAL NUMBER OF CLUSTERS
# ==========================================

if len(selected_features) >= 3:
    print("\n🔄 Finding optimal number of clusters...")
    
    inertia = []
    silhouette_scores = []
    davies_bouldin_scores = []
    K_range = range(2, 9)
    
    for k in K_range:
        print(f"   Testing K={k}...", end=" ")
        
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(X_scaled)
        
        inertia.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(X_scaled, cluster_labels))
        davies_bouldin_scores.append(davies_bouldin_score(X_scaled, cluster_labels))
        
        print(f"Silhouette: {silhouette_scores[-1]:.3f}")
    
    # Find optimal K
    best_k_silhouette = K_range[np.argmax(silhouette_scores)]
    best_k_davies = K_range[np.argmin(davies_bouldin_scores)]
    
    print(f"\n📊 Optimal K suggestions:")
    print(f"   Best Silhouette Score: K = {best_k_silhouette} (score: {max(silhouette_scores):.3f})")
    print(f"   Best Davies-Bouldin: K = {best_k_davies} (score: {min(davies_bouldin_scores):.3f})")
    
    # Use the K with best silhouette score
    optimal_k = best_k_silhouette
    print(f"\n🎯 Using K = {optimal_k} for final clustering")
    
    # --- Visualization of metrics ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Elbow plot
    axes[0].plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
    axes[0].set_title('📈 Elbow Method (Inertia)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Number of Clusters (K)')
    axes[0].set_ylabel('Inertia')
    axes[0].grid(True, alpha=0.3)
    
    # Silhouette scores
    axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
    axes[1].axvline(x=best_k_silhouette, color='red', linestyle='--', alpha=0.7, label=f'Best K={best_k_silhouette}')
    axes[1].set_title('📊 Silhouette Scores (Higher = Better)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Number of Clusters (K)')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Davies-Bouldin scores
    axes[2].plot(K_range, davies_bouldin_scores, 'ro-', linewidth=2, markersize=8)
    axes[2].axvline(x=best_k_davies, color='green', linestyle='--', alpha=0.7, label=f'Best K={best_k_davies}')
    axes[2].set_title('📉 Davies-Bouldin Scores (Lower = Better)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Number of Clusters (K)')
    axes[2].set_ylabel('Davies-Bouldin Score')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ Cannot proceed with clustering - insufficient features")
    optimal_k = None


🔄 Finding optimal number of clusters...
   Testing K=2... Silhouette: 0.717
   Testing K=3... Silhouette: 0.688
   Testing K=4... Silhouette: 0.266
   Testing K=5... Silhouette: 0.276
   Testing K=6... Silhouette: 0.294
   Testing K=7... Silhouette: 0.305
   Testing K=8... 

In [ ]:
# ==========================================
# 🚀 FINAL K-MEANS SEGMENTATION
# ==========================================

if optimal_k is not None:
    print(f"\n🚀 Performing final clustering with K = {optimal_k}...")
    
    # Final clustering
    final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
    cluster_labels_clean = final_kmeans.fit_predict(X_scaled)
    
    # Map results back to original dataset
    # Create a full array with -1 for outliers
    cluster_labels_full = np.full(len(df), -1)
    cluster_labels_full[outlier_mask] = cluster_labels_clean
    
    # Assign outliers to nearest cluster
    if outliers_removed > 0:
        print(f"🔄 Assigning {outliers_removed:,} outliers to nearest clusters...")
        X_outliers = X_imputed[~outlier_mask]
        X_outliers_scaled = scaler.transform(X_outliers)
        outlier_predictions = final_kmeans.predict(X_outliers_scaled)
        cluster_labels_full[~outlier_mask] = outlier_predictions
    
    # Add cluster labels to dataframe
    df['Segment_ID'] = cluster_labels_full
    df['Segment'] = 'Segment ' + (df['Segment_ID'] + 1).astype(str)
    
    # Calculate final metrics
    final_silhouette = silhouette_score(X_scaled, cluster_labels_clean)
    final_davies_bouldin = davies_bouldin_score(X_scaled, cluster_labels_clean)
    
    print(f"\n✅ Clustering completed successfully!")
    print(f"📊 Final cluster quality:")
    print(f"   Silhouette Score: {final_silhouette:.3f}")
    print(f"   Davies-Bouldin Score: {final_davies_bouldin:.3f}")
    
    # Show segment distribution
    segment_distribution = df['Segment'].value_counts().sort_index()
    print(f"\n📈 Segment Distribution:")
    for segment, count in segment_distribution.items():
        percentage = (count / len(df)) * 100
        print(f"   {segment}: {count:,} products ({percentage:.1f}%)")
    
else:
    print("❌ Cannot perform clustering - no optimal K found")

In [ ]:
# ==========================================
# 📊 SEGMENT ANALYSIS & PROFILING
# ==========================================

if optimal_k is not None:
    print("\n📊 Analyzing segment characteristics...")
    
    # Calculate segment profiles
    segment_profiles = df.groupby('Segment')[selected_features].agg(['mean', 'median', 'count']).round(2)
    
    # Show detailed segment analysis
    print(f"\n🔍 Detailed Segment Analysis:")
    print("=" * 80)
    
    for segment in sorted(df['Segment'].unique()):
        segment_data = df[df['Segment'] == segment]
        print(f"\n🎯 {segment} ({len(segment_data):,} products):")
        
        for feature in selected_features:
            mean_val = segment_data[feature].mean()
            median_val = segment_data[feature].median()
            print(f"   {feature}:")
            print(f"      Mean: {mean_val:,.2f}")
            print(f"      Median: {median_val:,.2f}")
    
    # Calculate relative performance
    print(f"\n📈 Segment Performance (Relative to Overall Average):")
    print("=" * 80)
    
    overall_means = df[selected_features].mean()
    
    for segment in sorted(df['Segment'].unique()):
        segment_data = df[df['Segment'] == segment]
        segment_means = segment_data[selected_features].mean()
        
        print(f"\n🏷️ {segment}:")
        for feature in selected_features:
            if overall_means[feature] != 0:
                ratio = segment_means[feature] / overall_means[feature]
                if ratio > 1.5:
                    status = "🔥 VERY HIGH"
                elif ratio > 1.2:
                    status = "📈 HIGH"
                elif ratio > 0.8:
                    status = "➖ AVERAGE"
                elif ratio > 0.5:
                    status = "📉 LOW"
                else:
                    status = "❄️ VERY LOW"
                print(f"   {feature}: {ratio:.2f}x ({status})")
            else:
                print(f"   {feature}: N/A (zero baseline)")
    
    # Create segment labels based on characteristics
    print(f"\n🏷️ Generating Business-Friendly Segment Labels...")
    
    segment_labels = {}
    for segment in sorted(df['Segment'].unique()):
        segment_data = df[df['Segment'] == segment]
        
        # Calculate relative scores
        revenue_score = segment_data['TotalRevenue'].mean() / overall_means['TotalRevenue'] if 'TotalRevenue' in selected_features else 1
        profit_score = segment_data['ProfitMargin'].mean() / overall_means['ProfitMargin'] if 'ProfitMargin' in selected_features else 1
        
        # Assign meaningful labels
        if revenue_score > 1.5 and profit_score > 1.2:
            label = "⭐ Premium Stars"
        elif revenue_score > 1.2:
            label = "🏆 Revenue Drivers"
        elif profit_score > 1.2:
            label = "💎 High-Margin Specialists"
        elif revenue_score < 0.7 and profit_score < 0.7:
            label = "⚠️ Underperformers"
        else:
            label = "📊 Steady Performers"
        
        segment_labels[segment] = label
        print(f"   {segment} → {label}")
    
    # Add business labels to dataframe
    df['Segment_Label'] = df['Segment'].map(segment_labels)
    
else:
    print("❌ Cannot perform segment analysis - clustering failed")

In [ ]:
# ==========================================
# 📊 VISUALIZATION DASHBOARD
# ==========================================

if optimal_k is not None:
    print("\n🎨 Creating visualization dashboard...")
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Segment Distribution (Pie Chart)
    ax1 = plt.subplot(3, 3, 1)
    segment_counts = df['Segment'].value_counts().sort_index()
    colors = plt.cm.Set3(np.linspace(0, 1, len(segment_counts)))
    wedges, texts, autotexts = ax1.pie(segment_counts.values, labels=segment_counts.index, 
                                       autopct='%1.1f%%', colors=colors, startangle=90)
    ax1.set_title('📊 Product Distribution by Segment', fontsize=14, fontweight='bold')
    
    # 2. Segment Sizes (Bar Chart)
    ax2 = plt.subplot(3, 3, 2)
    bars = ax2.bar(segment_counts.index, segment_counts.values, color=colors, alpha=0.8)
    ax2.set_title('📈 Segment Sizes', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Number of Products')
    ax2.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 50,
                 f'{int(height):,}', ha='center', va='bottom', fontweight='bold')
    
    # 3. PCA Visualization
    ax3 = plt.subplot(3, 3, 3)
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # Create color map for clusters
    unique_labels = sorted(np.unique(cluster_labels_clean))
    colors_scatter = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
    
    for i, label in enumerate(unique_labels):
        mask = cluster_labels_clean == label
        ax3.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=[colors_scatter[i]], label=f'Segment {label+1}', alpha=0.6, s=30)
    
    ax3.set_title('🎯 Clusters in PCA Space', fontsize=14, fontweight='bold')
    ax3.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    ax3.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 4-6. Feature comparison across segments (if we have key features)
    key_viz_features = ['TotalRevenue', 'ProfitMargin', 'SalesQty_sum']
    available_viz_features = [f for f in key_viz_features if f in selected_features]
    
    for idx, feature in enumerate(available_viz_features[:3]):
        ax = plt.subplot(3, 3, 4 + idx)
        feature_by_segment = df.groupby('Segment')[feature].mean()
        bars = ax.bar(feature_by_segment.index, feature_by_segment.values, 
                     color=colors, alpha=0.8)
        ax.set_title(f'📊 Average {feature} by Segment', fontsize=12, fontweight='bold')
        ax.set_ylabel(feature)
        ax.tick_params(axis='x', rotation=45)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:,.0f}', ha='center', va='bottom', fontsize=9)
    
    # 7. Segment Labels Distribution
    ax7 = plt.subplot(3, 3, 7)
    label_counts = df['Segment_Label'].value_counts()
    bars = ax7.bar(range(len(label_counts)), label_counts.values, 
                  color=plt.cm.viridis(np.linspace(0, 1, len(label_counts))), alpha=0.8)
    ax7.set_title('🏷️ Business Segment Labels', fontsize=12, fontweight='bold')
    ax7.set_ylabel('Number of Products')
    ax7.set_xticks(range(len(label_counts)))
    ax7.set_xticklabels(label_counts.index, rotation=45, ha='right')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax7.text(bar.get_x() + bar.get_width()/2., height + 50,
               f'{int(height):,}', ha='center', va='bottom', fontsize=9)
    
    # 8. Cluster Quality Metrics
    ax8 = plt.subplot(3, 3, 8)
    metrics_names = ['Silhouette\nScore', 'Davies-Bouldin\nScore']
    metrics_values = [final_silhouette, final_davies_bouldin]
    colors_metrics = ['green' if final_silhouette > 0.3 else 'orange', 
                     'green' if final_davies_bouldin < 1.0 else 'orange']
    
    bars = ax8.bar(metrics_names, metrics_values, color=colors_metrics, alpha=0.7)
    ax8.set_title('📈 Cluster Quality Metrics', fontsize=12, fontweight='bold')
    ax8.set_ylabel('Score')
    
    # Add value labels
    for bar, value in zip(bars, metrics_values):
        height = bar.get_height()
        ax8.text(bar.get_x() + bar.get_width()/2., height + 0.01,
               f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 9. Summary Statistics
    ax9 = plt.subplot(3, 3, 9)
    ax9.axis('off')
    
    summary_text = f"""
📊 SEGMENTATION SUMMARY
========================
Total Products: {len(df):,}
Number of Segments: {optimal_k}
Features Used: {len(selected_features)}
Outliers Removed: {outliers_removed:,}

🎯 CLUSTER QUALITY
Silhouette Score: {final_silhouette:.3f}
Davies-Bouldin: {final_davies_bouldin:.3f}

📈 LARGEST SEGMENT
{segment_counts.index[0]}: {segment_counts.iloc[0]:,} products
({segment_counts.iloc[0]/len(df)*100:.1f}%)

💎 BUSINESS SEGMENTS
{len(df['Segment_Label'].unique())} unique labels created
Most common: {df['Segment_Label'].mode().iloc[0]}
"""
    
    ax9.text(0.05, 0.95, summary_text, transform=ax9.transAxes, fontsize=11,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    
    plt.tight_layout()
    plt.suptitle('🎯 Product Segmentation Analysis Dashboard', fontsize=18, fontweight='bold', y=0.98)
    plt.show()
    
else:
    print("❌ Cannot create visualizations - clustering failed")

In [ ]:
# ==========================================
# 💾 EXPORT RESULTS
# ==========================================

if optimal_k is not None:
    print("\n💾 Exporting segmentation results...")
    
    # Export main segmentation file
    segmentation_output = df[['ProductID', 'Segment_ID', 'Segment', 'Segment_Label']].copy()
    segmentation_output = segmentation_output.sort_values(['Segment_ID', 'ProductID']).reset_index(drop=True)
    segmentation_output.to_csv('fixed_product_segmentation.csv', index=False)
    
    # Export complete dataset with segments
    df.to_csv('products_with_fixed_segments.csv', index=False)
    
    # Export segment profiles
    segment_summary = df.groupby(['Segment', 'Segment_Label'])[selected_features].agg(['mean', 'median', 'count'])
    segment_summary.to_csv('fixed_segment_profiles.csv')
    
    # Create comprehensive summary report
    summary_report = f"""
🔧 Fixed Product Segmentation Analysis Summary
==============================================
Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
Total Products Analyzed: {len(df):,}
Number of Segments: {optimal_k}

🛠️ DATA PREPROCESSING:
- Features Used: {len(selected_features)} ({', '.join(selected_features)})
- Missing Value Strategy: Median imputation
- Outlier Removal: {outliers_removed:,} products ({outliers_removed/len(df)*100:.1f}%)
- Scaling Method: StandardScaler

📊 CLUSTERING RESULTS:
- Algorithm: K-Means with {optimal_k} clusters
- Initialization: k-means++ with 20 random starts
- Silhouette Score: {final_silhouette:.3f} {'(Good)' if final_silhouette > 0.3 else '(Fair)' if final_silhouette > 0.2 else '(Poor)'}
- Davies-Bouldin Score: {final_davies_bouldin:.3f} {'(Good)' if final_davies_bouldin < 1.0 else '(Fair)' if final_davies_bouldin < 1.5 else '(Poor)'}

📈 SEGMENT DISTRIBUTION:
"""
    
    for segment, count in segment_distribution.items():
        percentage = (count / len(df)) * 100
        label = segment_labels.get(segment, 'Unknown')
        summary_report += f"- {segment}: {count:,} products ({percentage:.1f}%) → {label}\n"
    
    summary_report += f"""
🎯 BUSINESS INSIGHTS:
"""
    
    # Add insights for each segment
    for segment in sorted(df['Segment'].unique()):
        segment_data = df[df['Segment'] == segment]
        label = segment_labels.get(segment, 'Unknown')
        
        # Calculate key metrics if available
        if 'TotalRevenue' in selected_features:
            avg_revenue = segment_data['TotalRevenue'].mean()
            revenue_ratio = avg_revenue / overall_means['TotalRevenue']
            revenue_insight = f"Avg Revenue: ${avg_revenue:,.2f} ({revenue_ratio:.2f}x overall)"
        else:
            revenue_insight = "Revenue data not available"
        
        summary_report += f"\n{segment} - {label}:\n"
        summary_report += f"  Size: {len(segment_data):,} products ({len(segment_data)/len(df)*100:.1f}%)\n"
        summary_report += f"  {revenue_insight}\n"
    
    summary_report += f"""

{'✅ High Quality' if final_silhouette > 0.4 else '⚠️ Moderate Quality' if final_silhouette > 0.25 else '❌ Low Quality'} - Silhouette Score: {final_silhouette:.3f}
{'✅ Well Separated' if final_davies_bouldin < 1.0 else '⚠️ Moderately Separated' if final_davies_bouldin < 1.5 else '❌ Poorly Separated'} - Davies-Bouldin Score: {final_davies_bouldin:.3f}
"""
    
    # Save summary report
    with open('fixed_segmentation_summary.txt', 'w') as f:
        f.write(summary_report)
    
    print("✅ All results exported successfully!")
    print(f"📊 Main segmentation: fixed_product_segmentation.csv")
    print(f"📈 Complete dataset: products_with_fixed_segments.csv")
    print(f"📋 Segment profiles: fixed_segment_profiles.csv")
    print(f"📄 Summary report: fixed_segmentation_summary.txt")
    
    # Display sample results
    print(f"\n📋 Sample Results:")
    print("=" * 60)
    print(segmentation_output.head(10).to_string(index=False))
    
    print(f"\n🎯 Final Segment Summary:")
    print("=" * 60)
    for segment, label in segment_labels.items():
        count = len(df[df['Segment'] == segment])
        percentage = (count / len(df)) * 100
        print(f"{segment}: {count:,} products ({percentage:.1f}%) → {label}")
    
    print(summary_report)
    
else:
    print("❌ Cannot export results - clustering failed")
    print("Please check the data and feature availability.")